# 01 — Exploratory Data Analysis
IEEE-CIS Fraud Detection

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
trans = pd.read_csv('../data/raw/train_transaction.csv', low_memory=False)
identity = pd.read_csv('../data/raw/train_identity.csv', low_memory=False)
df = trans.merge(identity, on='TransactionID', how='left')
print(df.shape)

In [ ]:
# Class imbalance
print(df['isFraud'].value_counts(normalize=True))

In [ ]:
# Missing values
null_pct = df.isnull().mean().sort_values(ascending=False)
print(null_pct[null_pct > 0.5].head(20))

In [ ]:
# Transaction amount distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['TransactionAmt'].hist(bins=100, ax=axes[0])
axes[0].set_title('Raw Amount')
np.log1p(df['TransactionAmt']).hist(bins=100, ax=axes[1])
axes[1].set_title('Log Amount')
plt.show()

In [ ]:
# Fraud rate by hour of day
df['hour'] = (df['TransactionDT'] / 3600).astype(int) % 24
df.groupby('hour')['isFraud'].mean().plot(kind='bar', figsize=(12, 3))
plt.title('Fraud Rate by Hour of Day')
plt.show()

In [ ]:
# Card velocity features
df['card_txn_count'] = df.groupby('card1')['TransactionID'].transform('count')
df['card_avg_amt'] = df.groupby('card1')['TransactionAmt'].transform('mean')
df['amt_vs_card_avg'] = df['TransactionAmt'] / (df['card_avg_amt'] + 1)
df.groupby('isFraud')[['amt_vs_card_avg', 'card_txn_count']].mean()